# 정적 웹페이지 수집하기
* 순수 HTML, CSS로 만들어진 페이지
* javascript로 내용을 갱신하지 않는 페이지

# Yes24 베스트셀러 자료 수집하기

In [6]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup as bs

In [ ]:
https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=24

In [7]:
url="https://www.yes24.com/product/category/bestseller"
payload=dict(categoryNumber="001",pageNumber=1,pageSize=120)
r=requests.get(url,params=payload)
print(r.url)
print(r.status_code)
soup=bs(r.content,'lxml')
time.sleep(5)


https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200


* 전체 yes24 베스트셀러 페이지 중 책 정보가 들어있는 곳: ul#yesBestList
* ul#yesBestList 아래의 li에 책 1권의 정보가 들어있음.

In [ ]:
# soup.select_one("ul#yesBestList").select("li")

In [8]:
# yes24 전체 페이지에서 책 정보가 들어있는 부분만 잘라서 book_list에 저장
book_list=soup.select("ul#yesBestList > li")
book_list[:2]

[<li class="" data-goods-no="144021119" data-iy-no="0" data-statgb="02">
 <div class="itemUnit">
 <div class="item_img">
 <div class="img_canvas">
 <div class="img_upper">
 <em class="ico rank">1</em>
 <span class="rank_info rank_even">
 <span class="ico"></span><em class="txt rank"></em><em class="txt">위 상승</em>
 </span>
 </div>
 <span class="img_item">
 <span class="img_grp">
 <a class="lnk_img" href="/product/goods/144021119" onclick="wiseLogV2('BS', '001_005_001', ''); ">
 <em class="img_bdr">
 <img alt="단 한 번의 삶" border="0" class="lazy" data-original="https://image.yes24.com/goods/144021119/L" src="https://image.yes24.com/momo/Noimg_L.jpg"/>
 </em>
 </a>
 </span>
 </span>
 </div>
 <div class="img_btn">
 <a class="btnC btn_preview" href="javascript:yes24GU.openPreviewCheck(144021119); wiseLogV2('BS', '001_005_011', '');"><span class="bWrap"><em class="txt">미리보기</em></span></a>
 </div>
 </div>
 <div class="item_info">
 <div class="info_row info_keynote">
 <span class="gd_keynote" id

In [9]:
result={}
for idx, book in enumerate(book_list):
    print(f"{idx}/{len(book_list)} 추출중", end="\r")
    # 책 제목
    book_title = book.select_one(".gd_name").text
    # 저자
    author = book.select_one(".info_row.info_pubGrp a").text
    # 출판사
    publisher = book.select_one(".authPub.info_pub a").text
    # 출간일
    date_pub = book.select_one(".authPub.info_date").text
    # 가격
    price = book.select_one(".info_row.info_price em.yes_b").text
    # 평점
    rating = book.select_one(".rating_grade em.yes_b").text if book.select_one(".rating_grade em.yes_b") != None else 0.0
    # 리뷰수
    n_reviews = book.select_one(".info_row.info_rating em.txC_blue").text if book.select_one(".info_row.info_rating em.txC_blue") != None else 0
    
    keys = ['book_title', 'author', 'publisher', 'date_pub', 'price', 'rating', 'n_reviews']
    values = [book_title, author, publisher, date_pub, price, rating, n_reviews]
    for key, value in zip(keys, values):
        result.setdefault(key, []).append(value)

for key, value in result.items():
    print(key, len(value))
    
df = pd.DataFrame(result)
df    

book_title 120
author 120
publisher 120
date_pub 120
price 120
rating 120
n_reviews 120


,book_title,author,publisher,date_pub,price,rating,n_reviews
0,단 한 번의 삶,김영하,복복서가,2025년 04월,"15,120",8.8,46
1,여학교의 별 4,와야마 야마,문학동네,2025년 03월,"7,650",10.0,21
2,듀얼 브레인,이선 몰릭,상상스퀘어,2025년 03월,"18,900",8.4,46
3,어른의 품격을 채우는 100일 필사 노트,김종원,청림Life,2025년 03월,"18,000",9.9,64
4,소년이 온다,한강,창비,2014년 05월,"13,500",9.7,"3,860"
...,...,...,...,...,...,...,...
115,구의 증명,최진영,은행나무,2023년 04월,"10,800",9.1,747
116,그 개와 혁명,예소연,다산책방,2025년 02월,"15,750",9.3,36
117,2025 이지패스 ADsP 데이터분석 준전문가,전용문,위키북스,2025년 01월,"27,000",10.0,92
118,죽이고 싶은 아이,이꽃님,우리학교,2021년 06월,"11,250",9.5,975


In [10]:
print("가격: ", len(list(soup.select("ul#yesBestList .info_row.info_price em.yes_b"))))
print("평점: ", len(list(soup.select("ul#yesBestList .rating_grade em.yes_b"))))
print("리뷰수: ", len(list(soup.select("ul#yesBestList .info_row.info_rating em.txC_blue"))))

가격:  120
평점:  114
리뷰수:  114


In [33]:
result['author'][23]

'ETS'

In [35]:
book_list[23].select_one(".authPub.info_auth").text.replace("\n", " ").replace("\r", " ").strip()

'ETS 저'

# 저자, 역자, 글그림, 편저, 공동저자등 구분하기

In [36]:
def text_clean(text):
    return text.replace("\n", " ").replace("\r", " ").strip()

In [37]:
def author_extraction(book):
    author = ""
    photo = ""
    trans = ""
    paint = ""
    for idx, item in enumerate(text_clean(book.select_one(".authPub.info_auth").text).split("/")):
        print(idx, item)
        if idx == 0:
            if "저" == item[-1]:
                author = text_clean(item[:-2])
            elif "글" == item[-1]:
                author = text_clean(item[:-2])
            elif "글그림" == item[-3:]:
                author = text_clean(item[:-4])
        else:        
            if "정보 더 보기" == item[-7:]:
                author = text_clean(item.replace("정보 더 보기", ""))
            elif "사진" == item[-2:]:
                photo = text_clean(item[:-3])
            elif "역" == item[-1]:
                trans = text_clean(item[:-2])
            elif "글그림" == item[-3:]:
                paint = text_clean(item[:-4])
            
    #print(f"author{author}, photo{photo}, trans{trans}, paint{paint}")
    return author, photo, trans, paint

In [38]:
url = "https://www.yes24.com/product/category/bestseller"
payload = dict(categoryNumber="001", pageNumber=1, pageSize=120)
r = requests.get(url, params=payload)
print(r.url)
print(r.status_code)
soup = bs(r.content, "lxml")
time.sleep(5)
book_list = soup.select("ul#yesBestList > li")
for book in book_list[:2]:
    author, photo, trans, paint = author_extraction(book)
    print(author, photo, trans, paint)

https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200
0 김영하 저
author김영하, photo, trans, paint
김영하   
0 와야마 야마 글그림
1 현승희 역
author와야마 야마, photo, trans현승희, paint
와야마 야마  현승희 


In [ ]:
for item in text_clean(book_list[20].select_one(".authPub.info_auth").text).split("/"):
#     print(item)
    if "감추기" in item[:4]:
        author = item.replace("감추기", "").strip()
        print(author)

# 전체 페이지 수집하기

In [39]:
result_list = []
page = 1
while True:
    url = "https://www.yes24.com/product/category/bestseller"
    payload = dict(categoryNumber="001", pageNumber=page, pageSize=120)
    r = requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    soup = bs(r.content, "lxml")
    time.sleep(5)
    # yes24 전체 페이지에서 책 정보가 들어있는 부분만 잘라서 book_list에 저장
    book_list = soup.select("ul#yesBestList > li")
    result = {}
    for idx, book in enumerate(book_list):
        print(f"{idx}/{len(book_list)} 추출중", end="\r")
        # 책 제목
        book_title = book.select_one(".gd_name").text
        # 저자
        author, photo, trans, paint = author_extraction(book)
        # 출판사
        publisher = book.select_one(".authPub.info_pub a").text
        # 출간일
        date_pub = book.select_one(".authPub.info_date").text
        # 가격
        price = book.select_one(".info_row.info_price em.yes_b").text
        # 평점
        rating = book.select_one(".rating_grade em.yes_b").text if book.select_one(".rating_grade em.yes_b") != None else 0.0
        # 리뷰수
        n_reviews = book.select_one(".info_row.info_rating em.txC_blue").text if book.select_one(".info_row.info_rating em.txC_blue") != None else 0

        keys = ['book_title', 'author', 'photo', 'trans', 'paint', 'publisher', 'date_pub', 'price', 'rating', 'n_reviews']
        values = [book_title, author, photo, trans, paint, publisher, date_pub, price, rating, n_reviews]
        for key, value in zip(keys, values):
            result.setdefault(key, []).append(value)

    for key, value in result.items():
        print(key, len(value))

    result_list.append(pd.DataFrame(result))
    
    if page < 10:
        page += 1
    else:
        break
    
result_list = pd.concat(result_list)
result_list = result_list.reset_index(drop=True)
result_list

https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=120
200
0 김영하 저출중
author김영하, photo, trans, paint
0 와야마 야마 글그림
1 현승희 역
author와야마 야마, photo, trans현승희, paint
0 이선 몰릭 저
1 신동숙 역
author이선 몰릭, photo, trans신동숙, paint
0 김종원 저출중
author김종원, photo, trans, paint
0 한강 저추출중
author한강, photo, trans, paint
0 태수 저추출중
author태수, photo, trans, paint
0 한강 저추출중
author한강, photo, trans, paint
0 양귀자 저출중
author양귀자, photo, trans, paint
0 코이케 류노스케 저
1 박재현 역
author코이케 류노스케, photo, trans박재현, paint
0 백온유, 강보라, 서장원, 성해나, 성혜령 저 외 2명                                           정보 더 보기
1 감추기    백온유 강보라 서장원 성해나 성혜령 이희주 현호정
author, photo, trans, paint
0 안녕달 글그림중
author안녕달, photo, trans, paint
0 최태성 저추출중
author최태성, photo, trans, paint
0 최태성 저추출중
author최태성, photo, trans, paint
0 김수미 저추출중
author김수미, photo, trans, paint
0 김주환 저추출중
author김주환, photo, trans, paint
0 유선경 저추출중
author유선경, photo, trans, paint
0 요시타케 신스케 글그림
1 권남희 역
author요시타케 신스케, photo, trans권남희, paint
0 유발 하라리 저
1 김명주 역
autho

0 ETS 저추출중
authorETS, photo, trans, paint
0 양귀자 저추출중
author양귀자, photo, trans, paint
0 대니얼 J. 시겔, 메리 하첼 저
1 신유희 역
author대니얼 J. 시겔, 메리 하첼, photo, trans신유희, paint
0 오은숙 글추출중
1 이국현 그림
author오은숙, photo, trans, paint
0 로이스 라우리 저
1 장은수 역
author로이스 라우리, photo, trans장은수, paint
0 ETS 저추출중
authorETS, photo, trans, paint
0 이주택 저추출중
author이주택, photo, trans, paint
0 쇼펜하우어 저 
1  김지민 편
author, photo, trans, paint
0 김승호 저추출중
author김승호, photo, trans, paint
0 권은희 저추출중
author권은희, photo, trans, paint
0 데이비드 이글먼 저
1 김승욱 역
author데이비드 이글먼, photo, trans김승욱, paint
0 전진권 저추출중
author전진권, photo, trans, paint
0 앙투안 드 생텍쥐페리 저
1 변광배 해설
2 김수영 역
author앙투안 드 생텍쥐페리, photo, trans김수영, paint
0 빨간내복야코 원저
1 전판교 글
2 도니패밀리 그림
3 샌드박스 네트워크 감수
author빨간내복야코, photo, trans, paint
0 일본어 공부기술연구소 저
author일본어 공부기술연구소, photo, trans, paint
0 이꽃님 저추출중
author이꽃님, photo, trans, paint
0 하마지 아키, 쿠미쵸 글그림
author하마지 아키, 쿠미쵸, photo, trans, paint
0 손무 원저추출중
1 임용한 편저
author손무, photo, trans, paint
0 David Cho 저
authorDavid Cho, photo, trans, paint
0 타

0 퍼시픽 학술편찬국 저
author퍼시픽 학술편찬국, photo, trans, paint
0 아닐 아난타스와미 저
1 노승영 역
author아닐 아난타스와미, photo, trans노승영, paint
0 김애란 저 추출중
author김애란, photo, trans, paint
0 박응용 저 추출중
author박응용, photo, trans, paint
0 Coco Uzuki 글그림
1 정효진 역
authorCoco Uzuki, photo, trans정효진, paint
0 이영방 편저추출중
author이영방, photo, trans, paint
0 김용성 저 추출중
author김용성, photo, trans, paint
0 살만 칸 저추출중
1 박세연 역
author살만 칸, photo, trans박세연, paint
0 노한동 저 추출중
author노한동, photo, trans, paint
0 한날 글그림추출중
author한날, photo, trans, paint
0 김민철 편저추출중
author김민철, photo, trans, paint
0 김민서 저 추출중
author김민서, photo, trans, paint
0 해커스 일본어연구소 저
author해커스 일본어연구소, photo, trans, paint
0 이호선 저 추출중
author이호선, photo, trans, paint
0 백희나 글그림출중
author백희나, photo, trans, paint
0 윤영빈, 서용욱, 김학배, 박인상 저
author윤영빈, 서용욱, 김학배, 박인상, photo, trans, paint
0 이유진 편저추출중
author이유진, photo, trans, paint
0 정은숙 저 추출중
author정은숙, photo, trans, paint
0 이광렬 저 추출중
author이광렬, photo, trans, paint
book_title 120
author 120
photo 120
trans 120
paint 120
publisher 120
date_pub 120
pri

0 유하다요컨텐츠개발팀 저
author유하다요컨텐츠개발팀, photo, trans, paint
0 마티아스 뇔케 저
1 이미옥 역
author마티아스 뇔케, photo, trans이미옥, paint
0 이은경 저추출중
author이은경, photo, trans, paint
0 하정훈 저추출중
author하정훈, photo, trans, paint
0 안녕달 글그림중
author안녕달, photo, trans, paint
0 성완 글 추출중
1 돌만 그림
author성완, photo, trans, paint
0 네빌 고다드 저
1 홍주연 역
author네빌 고다드, photo, trans홍주연, paint
0 모리시타 미유 글그림
author모리시타 미유, photo, trans, paint
0 고희정 글추출중
1 조승연 그림
2 류정민 감수
author고희정, photo, trans, paint
0 최재천 저추출중
author최재천, photo, trans, paint
0 고종훈 저추출중
author고종훈, photo, trans, paint
0 김신지 저추출중
author김신지, photo, trans, paint
0 수미숨(상의민), 애나정 저
author수미숨(상의민), 애나정, photo, trans, paint
0 금준경 저추출중
1 방상호 그림
author금준경, photo, trans, paint
0 존 윌리엄스 저
1 김승욱 역
author존 윌리엄스, photo, trans김승욱, paint
0 앤절라 더크워스 저
1 김미정 역
author앤절라 더크워스, photo, trans김미정, paint
0 리사 리드센 저
1 손화수 역
author리사 리드센, photo, trans손화수, paint
0 최승필 저추출중
author최승필, photo, trans, paint
0 태쁘 원저추출중
1 김혜련 글
2 이소연 그림
3 샌드박스 네트워크 감수
author태쁘, photo, trans, paint
0 버트런드 러셀 저
1 장석봉 역
author

0 홍태성, 영진정보연구소 저
author홍태성, 영진정보연구소, photo, trans, paint
0 추공 원저 추출중
1 장성락(REDICE STUDIO) 글그림
author추공, photo, trans, paint장성락(REDICE STUDIO)
0 송주연, 김지학, 황혜림 저
author송주연, 김지학, 황혜림, photo, trans, paint
0 김정호 저 추출중
author김정호, photo, trans, paint
0 권경배 저 추출중
author권경배, photo, trans, paint
0 히가시노 게이고 저
1 양윤옥 역
author히가시노 게이고, photo, trans양윤옥, paint
0 이라일라 글추출중
1 박현주 그림
author이라일라, photo, trans, paint
0 별다름, 달다름 글
1 서영 그림
author별다름, 달다름, photo, trans, paint
0 ETS 저 추출중
authorETS, photo, trans, paint
0 이치조 미사키 저
1 권영주 역
author이치조 미사키, photo, trans권영주, paint
0 헤르만 헤세 저중
author헤르만 헤세, photo, trans, paint
0 정유정 저 추출중
author정유정, photo, trans, paint
0 박준 저0 추출중
author박준, photo, trans, paint
0 제롬 데이비드 샐린저 저
1 정영목 역
author제롬 데이비드 샐린저, photo, trans정영목, paint
0 조주희 글 추출중
1 김정한 그림
2 김미영 기획
author조주희, photo, trans, paint
0 조승리 저 추출중
author조승리, photo, trans, paint
0 라이쿠 마코토 글그림
author라이쿠 마코토, photo, trans, paint
0 프란치스코 교황, 카를로 무쏘 저
1 이재협, 김호열, 이창욱 역 외 1명                                           정보 더 보

0 해커스 취업교육연구소 저
author해커스 취업교육연구소, photo, trans, paint
0 알랭 드 보통 저
1 정영목 역
author알랭 드 보통, photo, trans정영목, paint
0 윤동주 저추출중
1 윤동주100년포럼 편
author윤동주, photo, trans, paint
0 이공편입수학연구소 저
author이공편입수학연구소, photo, trans, paint
0 이나모리 가즈오 저
1 김윤경 역
author이나모리 가즈오, photo, trans김윤경, paint
0 최지혜 글추출중
1 김소라(김고둥) 그림
author최지혜, photo, trans, paint
0 마르쿠스 아우렐리우스 저
1 박문재 역
author마르쿠스 아우렐리우스, photo, trans박문재, paint
0 J. 켄지 로페즈-알트 저
1 임현수 역
2 송윤형 감수
authorJ. 켄지 로페즈-알트, photo, trans임현수, paint
0 마이클 샌델 저
1 안기순 역
2 김선욱 감수
author마이클 샌델, photo, trans안기순, paint
0 셸 실버스타인 글그림
author셸 실버스타인, photo, trans, paint
0 주디스 올로프 저
1 김현정 역
author주디스 올로프, photo, trans김현정, paint
0 고성준 저추출중
author고성준, photo, trans, paint
0 스즈키 유스케 저
1 명다인 역
author스즈키 유스케, photo, trans명다인, paint
0 이기문 저추출중
author이기문, photo, trans, paint
0 제임스 후퍼, 강민아 저
author제임스 후퍼, 강민아, photo, trans, paint
0 레누카 가브라니 저
1 최유경 역
author레누카 가브라니, photo, trans최유경, paint
0 SDC 편저출중
authorSDC, photo, trans, paint
0 전우성 저추출중
author전우성, photo, trans, paint
0 최대호 저추

,book_title,author,photo,trans,paint,publisher,date_pub,price,rating,n_reviews
0,단 한 번의 삶,김영하,,,,복복서가,2025년 04월,"15,120",8.8,46
1,여학교의 별 4,와야마 야마,,현승희,,문학동네,2025년 03월,"7,650",10.0,21
2,듀얼 브레인,이선 몰릭,,신동숙,,상상스퀘어,2025년 03월,"18,900",8.4,46
3,어른의 품격을 채우는 100일 필사 노트,김종원,,,,청림Life,2025년 03월,"18,000",9.9,64
4,소년이 온다,한강,,,,창비,2014년 05월,"13,500",9.7,"3,860"
...,...,...,...,...,...,...,...,...,...,...
994,2025 에듀윌 공인중개사 한영규 합격서 부동산세법,한영규,,,,에듀윌,2025년 01월,"14,400",9.4,7
995,2025 에듀윌 공인중개사 김민석 합격서 부동산공시법,김민석,,,,에듀윌,2025년 01월,"15,300",9.7,7
996,미로 속 아이,기욤 뮈소,,양영란,,밝은세상,2024년 12월,"16,650",9.2,117
997,김수연의 아기발달 백과,김수연,,,,삼인,2024년 11월,"25,200",10.0,12
